In [1]:
import os

In [2]:
!pwd

/Users/abhikchoudhury/Library/CloudStorage/OneDrive-IBM/Badlo/Krish Naik Projects/Kidney_Tumor_Identification_System_Computer_vision/Kidney Tumor Identification System_Abhik/research


In [ ]:
#Going one level back where I need to create the project directory( in the main Kidney Tumor Identification System_Abhik folder)
os.chdir("../")

In [4]:
pwd

'/Users/abhikchoudhury/Library/CloudStorage/OneDrive-IBM/Badlo/Krish Naik Projects/Kidney_Tumor_Identification_System_Computer_vision/Kidney Tumor Identification System_Abhik'

In [ ]:
#Here we have entity. We update it here for the the time bieng since we are experimenting here. 
#Later we will add the entities to the src/entity folder when we piece togather the modular components. 

from dataclasses import dataclass
from pathlib import Path
#Here entity DataIngestionConfig is the return type of the data ingestion function defined in config.yaml Eg Root_dir return path type,
#source_url returns string type etc.
#dataclass is a decorator assigned on top of any python class to treat it as an entity. Instead of the class DataIngestionConfig 
#becoming a class, due to @dataclass decorator, the class DataIngestionConfig is treated as a class variable / entity. 
#The purpose of creating this entity is to ensude that an function is restricted to return only a specific data type and nothing else. This entity has been used as 
#a return type later in the code in get_data_ingestion_config().

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path


#After this notebook is run, then this entity is moved to cnnclassifier/config_entity.py

In [ ]:
#we are importing the params.yaml and config.yaml path from the cnnclassifier/constants folder.
from cnnClassifier.constants import *
#Also import the functions that we creatd from cnnclassifier/utils/common program
from cnnClassifier.utils.common import read_yaml, create_directories

In [ ]:
#we need to update the configuration manager in src config. For this , we will have to update the configuration.py within 
# cnnClassifier/config. 
#However, we will make this update later when our test coding is done. For now, we will update the configuration manager in 
# the 01_data_ingestion.ipynb itself. 
#We do that by creating a class ConfigurationManager. 

#here CONFIG_FILE_PATH and PARAMS_FILE_PATH come from the init.py of cnnClassifier/contants and stored int eh class 
# variables config_filepath and params_filepath. from the read_yaml(), which are defined in cnnclassifier/utils/common program, 
# we read the config.yaml and params.paml files
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
#Now it will read the artifacts_root from config.yaml and pass it to the create_directiories() 
# function call (create_directories defined in commons.py)
#and create the directories. local class var self.configsnow already has the artifacts_root value, 
# which is nothing but a folder name. 
        create_directories([self.config.artifacts_root])


#here we prepare data ingestion related configurations. Check the return type,DataIngestionConfig which was defined earlier 
# in this code. This is the standard practice. 
    
    def get_data_ingestion_config(self) -> DataIngestionConfig:

#   data_ingestion defined in config.yaml and all the values(key value pair of the dictionary) is stores within the class 
# variable config and used subsequently.      
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        #returned using the DataIngestionConfig entity defined earlier. 
        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir 
        )

        return data_ingestion_config

#Once this notebook is run, we will paste this configuration manaher code in cnnClassifier/config/configuration.py

In [9]:
import os
import urllib.request as request
import zipfile
import gdown
from cnnClassifier import logger
from cnnClassifier.utils.common import get_size

In [ ]:
#Now we need to update the data ingestion components. For the time being, we will 
#update it in the 01_data_ingestion.ipynb only, and will later do it in the respective project folder. 
#First we create data ingestion class, which will take dataingestionConfig which was defined inside 
#class ConfigurationManager in the get_data_ingestion_config(), which will give you access to root_dir, source_url,
# local_data_file and unzip_dir using the selg.config.
#The self.config's source_URL and local_data_file variables are used to fetch the data from Google drive and save to your project. 
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    
    def download_file(self)-> str:
        '''
        Fetch data from the url 
        '''

        try: 
            dataset_url = self.config.source_URL
            zip_download_dir = self.config.local_data_file
            os.makedirs("artifacts/data_ingestion", exist_ok=True)
            logger.info(f"Downloading data from {dataset_url} into file {zip_download_dir}")

            file_id = dataset_url.split("/")[-2]
            prefix = 'https://drive.google.com/uc?/export=download&id='
            gdown.download(prefix+file_id,zip_download_dir)

            logger.info(f"Downloaded data from {dataset_url} into file {zip_download_dir}")

        except Exception as e:
            raise e

#here the Zip file is extracted,
    
    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)
#Once this notebook is run, we will put this whole data ingestion component in the cnnClassfier/component/data_ingestion.py

In [ ]:
#Now we have to update the pipeline. However for testing, instead we will do it here. calling the requisite functions sequentially. 
#Once you run this cell, you will see the artifacts created the way it was defined in config.yaml. 
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

#Once this notebook is run, we will put this pipeline component in the cnnClassifier/pipeline/stage_02_prepare_base_model.py

[2026-05-03 12:51:25,117: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-05-03 12:51:25,121: INFO: common: yaml file: params.yaml loaded successfully]
[2026-05-03 12:51:25,122: INFO: common: created directory at: artifacts]
[2026-05-03 12:51:25,123: INFO: common: created directory at: artifacts/data_ingestion]
[2026-05-03 12:51:25,123: INFO: 3647029362: Downloading data from https://drive.google.com/file/d/1vlhZ5c7abUKF8xXERIw6m9Te8fW7ohw3/view?usp=sharing into file artifacts/data_ingestion/data.zip]


Downloading...
From (original): https://drive.google.com/uc?/export=download&id=1vlhZ5c7abUKF8xXERIw6m9Te8fW7ohw3
From (redirected): https://drive.google.com/uc?%2Fexport=download&id=1vlhZ5c7abUKF8xXERIw6m9Te8fW7ohw3&confirm=t&uuid=3d8bb85c-c911-4eb0-8f70-cc968162d96d
To: /Users/abhikchoudhury/Library/CloudStorage/OneDrive-IBM/Badlo/Krish Naik Projects/Kidney_Tumor_Identification_System_Computer_vision/Kidney Tumor Identification System_Abhik/artifacts/data_ingestion/data.zip
100%|██████████| 57.7M/57.7M [00:06<00:00, 9.24MB/s]

[2026-05-03 12:51:33,733: INFO: 3647029362: Downloaded data from https://drive.google.com/file/d/1vlhZ5c7abUKF8xXERIw6m9Te8fW7ohw3/view?usp=sharing into file artifacts/data_ingestion/data.zip]
